# Automated AI Bakery Verification

In this notebook, the procedure developed in the exploratory verification pilot will be applied with those findings. The bakery candidates will be processed in descending ranks, with each verifcation result saved and considered for the overall classification. Businesses will be processed in API batches and the process will be resumed across multiple days due to API rate limits. Previous verification results are loaded automatically so that businesses with completed verification are not resubmitted. The results will later be included in a final bakery name classification results csv.

In [37]:
from pathlib import Path
from getpass import getpass
from datetime import datetime

import json
import time
import pandas as pd

from google import genai
from google.genai import types, errors

# Verification settings
MODEL = "gemini-2.5-flash"

BUSINESSES_PER_REQUEST = 10
MAX_REQUESTS_PER_RUN = 1
REQUEST_DELAY_SECONDS = 2

# Paths
FHRS_DATA_DATE = "2026-07-23"

DATA_FOLDER = Path("../data/business/interim")
VERIFICATION_FOLDER = (DATA_FOLDER / "ai_verification")
VERIFICATION_FOLDER.mkdir(parents=True, exist_ok=True)

RANKED_FHRS_PATH = (DATA_FOLDER / f"london_fhrs_ranked_establishments_{FHRS_DATA_DATE}.csv")
RAW_FHRS_PATH = Path(f"../data/business/raw/london_fhrs_raw_{FHRS_DATA_DATE}.csv")

RESULTS_PATH = (VERIFICATION_FOLDER / "bakery_ai_verification_results_v5_final.csv")
BATCH_LOG_PATH = (VERIFICATION_FOLDER / "bakery_ai_verification_batches_v5_final.jsonl")
ERROR_LOG_PATH = (VERIFICATION_FOLDER / "bakery_ai_verification_errors_v5_final.jsonl")

# Loading dataset and checking recall range

Since verification via Gemini API can be costly, I found that choosing a recall threshold based on the classifier findings would be a more reasonable way to verify the business names.

In [3]:
fhrs_ranked = pd.read_csv(RANKED_FHRS_PATH)

address_columns = ["FHRSID", "AddressLine1", "AddressLine2", "AddressLine3", "AddressLine4"]

fhrs_raw_address = pd.read_csv(RAW_FHRS_PATH, usecols=address_columns)

fhrs_raw_address = (fhrs_raw_address.drop_duplicates(subset="FHRSID"))

fhrs_ranked = fhrs_ranked.merge(fhrs_raw_address, on="FHRSID", how="left", validate="many_to_one")

fhrs_ranked["Address"] = fhrs_ranked[["AddressLine1", "AddressLine2", "AddressLine3", "AddressLine4"]
                                     ].apply(lambda row: ", ".join(str(value).strip()
                                             for value in row 
                                             if pd.notna(value) and str(value).strip()), 
                                             axis=1)

verification_candidates = (
    fhrs_ranked[fhrs_ranked["BakeryRank"].notna()]
    .sort_values(["BakeryRank", "FHRSID"])
    .drop_duplicates(subset="BusinessNameClean")
    .rename(columns={"FHRSID": "FHRSIDRep"})
    .reset_index(drop=True))

# Candidate-selection threshold from classifier cross-validation (notebook_07)
RECALL_THRESHOLDS = {
    0.90: 0.021,
    0.95: 0.015,
    0.99: 0.006}

threshold_summary = pd.DataFrame({
    "TargetRecall": RECALL_THRESHOLDS.keys(),
    "BakeryScoreThreshold": RECALL_THRESHOLDS.values()
})

threshold_summary["Candidates"] = [(verification_candidates["BakeryScore"] >= threshold).sum()
                                   for threshold in threshold_summary["BakeryScoreThreshold"]]

threshold_summary["CandidateIncreaseFrom90%"] = ((
    threshold_summary["Candidates"] / (threshold_summary.loc[threshold_summary["TargetRecall"] == 0.90,"Candidates"].iloc[0]) - 1
    )* 100).round(2)

threshold_summary["RecallGainFrom90%"] = (threshold_summary["TargetRecall"] - 0.90) * 100

display(threshold_summary)


,TargetRecall,BakeryScoreThreshold,Candidates,CandidateIncreaseFrom90%,RecallGainFrom90%
0,0.90,0.021,19902,0.00,0.0
1,0.95,0.015,28847,44.95,5.0
2,0.99,0.006,49147,146.95,9.0


From this we can see that from 90% recall to 95% recall we add ~45% more candidates for potenitally 5% improvement in recall. Meanwhile for 9% increase from 90% recall to 99% recall we would add ~147% of candidates, which seems unreasonable as we are more than doubling the candidates for a potential improvement in recall

In [4]:
TARGET_RECALL = 0.95
MIN_BAKERY_SCORE = RECALL_THRESHOLDS[TARGET_RECALL]

verification_queue = (verification_candidates[verification_candidates["BakeryScore"] >= MIN_BAKERY_SCORE]
    .sort_values("BakeryRank")
    .reset_index(drop=True)
)

print(f"""
Selected target recall: {TARGET_RECALL:.0%}
BakeryScore threshold: {MIN_BAKERY_SCORE}
Businesses to verify: {len(verification_queue)}""")


Selected target recall: 95%
BakeryScore threshold: 0.015
Businesses to verify: 28847


In [38]:
if RESULTS_PATH.exists():
    previous_results = pd.read_csv(RESULTS_PATH)
    completed_names = set(previous_results["BusinessNameClean"].dropna())

else:
    previous_results = pd.DataFrame()
    completed_names = set()

remaining_queue = verification_queue[~verification_queue["BusinessNameClean"].isin(completed_names)].copy()
remaining_queue = (remaining_queue.sort_values("BakeryRank").reset_index(drop=True))

max_businesses_per_run = (BUSINESSES_PER_REQUEST * MAX_REQUESTS_PER_RUN)

run_queue = (remaining_queue
             .head(max_businesses_per_run)
             .copy()
             .reset_index(drop=True))

requests_planned = (len(run_queue) + BUSINESSES_PER_REQUEST- 1) // BUSINESSES_PER_REQUEST

print(
f"""Previously completed: {len(completed_names)}
Remaining businesses: {len(remaining_queue)}
Businesses selected this run: {len(run_queue)}
Maximum API requests this run: {requests_planned}""")

Previously completed: 28837
Remaining businesses: 10
Businesses selected this run: 10
Maximum API requests this run: 1


In [8]:
api_key = getpass("Gemini API key: ")

client = genai.Client(api_key=api_key)

In [29]:
def clean_value(value):
    if pd.isna(value):
        return "Unknown"
    return str(value)

def build_verification_prompt(batch):

    batch = batch.reset_index(drop=True)

    business_blocks = []

    for i, row in batch.iterrows():
        business_blocks.append(
            f"""
BUSINESS {i + 1}
Name: {clean_value(row["BusinessName"])}
Address: {clean_value(row["Address"])}
Postcode: {clean_value(row["PostCode"])}
Local authority: {clean_value(row["LocalAuthorityName"])}
FHRS type: {clean_value(row["BusinessType"])}
""".strip()
        )

    business_text = "\n\n".join(business_blocks)

    prompt = f"""
ROLE

You are an evidence-based researcher classifying London food businesses.

ACTION

Use Google Search to investigate all {len(batch)} businesses below.

For each business:

1. decide whether the web evidence reasonably matches the supplied London business;
2. classify it as BAKERY or NOT_BAKERY whenever the available evidence supports
   a reasonable conclusion.

Use UNCLEAR only when the business genuinely cannot be identified or its activity
cannot reasonably be determined.

Use enough searches to establish the business identity and actual activity.

If the first search is inconclusive, gives a conflicting location, or finds only
similarly named businesses elsewhere, try at least one alternative search using
the postcode, address, company name or local authority before returning UNCLEAR.

Do not continue searching once reliable evidence is sufficient for a decision.

CONTEXT

The classifications are for a dissertation measuring bakery provision in London.

A BAKERY is a business whose own commercial activity involves making or
specialising in bakery products such as bread, pastries, cakes, biscuits,
cookies, pies, doughnuts or similar baked products.

This includes bakeries, patisseries, cake makers/studios, bread bakeries,
micro-bakeries, wholesale bakeries and bakery-cafes.

A business does not have to sell only baked goods to qualify.

LOCATION MATCH

Return YES when the external evidence reasonably refers to the supplied business.

A full matching address or postcode is strong evidence.

When FHRS provides no street address or only a partial postcode, an exact or
distinctive business name operating in the same London borough or postcode area
is enough for YES when there is no conflicting location evidence.

A full street address is not required when the FHRS location has been suppressed.

Use UNCLEAR for location only when there are multiple plausible businesses,
the name is too generic to connect to one local business, or the evidence points
to a conflicting location.

CLASSIFICATION

Choose BAKERY when reliable evidence shows that the matched business:

- makes bakery products itself; OR
- specialises in selling bakery products as a substantial part of its business.

Choose NOT_BAKERY when the matched business primarily operates as something
else and bakery products are only incidental, minor, or absent.

Examples of NOT_BAKERY include:

- ordinary restaurants or cafes;
- supermarkets, grocery shops and convenience stores;
- general dessert parlours;
- sweet shops and chocolatiers;
- caterers;
- pizza restaurants or takeaways;
- schools, nurseries, care homes, hotels and pubs;
- baking-ingredient or cake-decorating supply shops.

Treat a supermarket as NOT_BAKERY even if it has a bakery aisle or sells
fresh bread.

Treat a baking-supply business as NOT_BAKERY when it sells ingredients,
equipment or decorations rather than bakery products it makes or specialises in.

For an identified business with clear non-bakery activity, choose NOT_BAKERY.
An external source does not need to explicitly say "not a bakery".

EVIDENCE

Prefer evidence in this order:

1. official website or menu;
2. official business social-media page or delivery platform;
3. Companies House activity or SIC information;
4. credible business directories, news or other business listings.

Use the evidence describing the business's actual products or services.
Specific product/activity evidence is stronger than a broad SIC code.

The business name and FHRS type help identify the business but are not enough
on their own to prove bakery activity.

If a matched business is closed, dissolved or dormant, still classify its
business type and begin the reason with "INACTIVE:".

EXAMPLES

Example A:
Official website identifies the same N16 business as a micro-bakery making
sourdough, focaccia and pastries.
→ YES | BAKERY

Example B:
The exact business is a supermarket selling groceries and fresh bread from
a bakery section.
→ YES | NOT_BAKERY

Example C:
A dessert shop menu mainly contains waffles, crepes, gelato and milkshakes,
plus one brownie or cake item.
→ YES | NOT_BAKERY

Example D:
FHRS gives only a partial postcode, but a distinctive same-name business in
the same borough has an official page showing bespoke cakes baked to order.
→ YES | BAKERY

Example E:
A generic name with a partial postcode matches several unrelated businesses
and there is no evidence identifying the London business.
→ UNCLEAR | UNCLEAR

OUTPUT

Return exactly {len(batch)} lines in this format:

business number | location match | verdict | short evidence reason

Example:
1 | YES | BAKERY | Official website says it makes sourdough and pastries daily.
2 | YES | NOT_BAKERY | Menu shows a general restaurant with only incidental cakes.
3 | UNCLEAR | UNCLEAR | Multiple businesses match the name and partial postcode.

Location match: YES or UNCLEAR
Verdict: BAKERY, NOT_BAKERY or UNCLEAR

Keep each reason to ONE short sentence of no more than 18 words.

Return every number from 1 to {len(batch)} exactly once.
Return only the result lines.

{business_text}
""".strip()

    return prompt

In [30]:
def verify_current_batch(batch):

    prompt = build_verification_prompt(batch)

    response = client.models.generate_content(
        model=MODEL,
        contents=prompt,
        config=types.GenerateContentConfig(
            tools=[types.Tool(google_search=types.GoogleSearch())],
            temperature=0,
            max_output_tokens=1000,
            thinking_config=types.ThinkingConfig(
                thinking_budget=0
            )
        )
    )

    return prompt, response

In [31]:
def get_grounding_info(response):

    if not response.candidates:
        return [], []

    grounding = response.candidates[0].grounding_metadata

    if grounding is None:
        return [], []

    search_queries = list(grounding.web_search_queries or [])

    source_urls = []

    for chunk in grounding.grounding_chunks or []:
        if chunk.web:
            source_urls.append(chunk.web.uri)

    source_urls = list(dict.fromkeys(source_urls))

    return search_queries, source_urls

In [32]:
def parse_response(response_text, batch, batch_id):

    batch = batch.reset_index(drop=True)

    text = (response_text.replace("\\_", "_").strip())

    # Gemini sometimes forgets the newlines between results so they're put back together before parsing.
    for number in range(len(batch), 0, -1):
        text = text.replace(f"{number} |", f"\n{number} |")

    parsed = {}

    for line in text.splitlines():
        parts = [part.strip()
                 for part in line.split("|", 3)]

        if len(parts) != 4:
            continue

        number_text = (parts[0].replace("*", "").strip())

        if not number_text.isdigit():
            continue

        business_number = int(number_text)

        if not 1 <= business_number <= len(batch):
            continue

        location_match = parts[1].upper()

        verdict = (parts[2].upper().replace(" ", "_"))

        if location_match not in {"YES", "UNCLEAR"}:
            continue

        if verdict not in {"BAKERY", "NOT_BAKERY", "UNCLEAR"}:
            continue

        if location_match == "UNCLEAR":
            verdict = "UNCLEAR"

        row = batch.iloc[business_number - 1]

        # Using the business number as the dictionary key so that repeated answers don't create duplicate rows.
        parsed[business_number] = {
            "BusinessNameClean": row["BusinessNameClean"],
            "BusinessName": row["BusinessName"],
            "FHRSIDRep": row["FHRSIDRep"],
            "BusinessType": row["BusinessType"],
            "PostCode": row["PostCode"],
            "LocalAuthorityName": row["LocalAuthorityName"],
            "Address": row["Address"],
            "BakeryRank": row["BakeryRank"],
            "BakeryScore": row["BakeryScore"],
            "StoreCount": row["StoreCount"],
            "LocationMatch": location_match,
            "AIVerdict": verdict,
            "AIReason": parts[3],
            "BatchID": batch_id,
            "Model": MODEL,
            "VerificationDateTime": datetime.now().isoformat(timespec="seconds")
            }

    missing_numbers = [number
                       for number in range(1, len(batch) + 1)
                       if number not in parsed]

    results = pd.DataFrame([parsed[number]
                            for number in sorted(parsed)])

    return results, missing_numbers


def save_results(results):

    if results.empty:
        return

    results.to_csv(RESULTS_PATH, mode="a", header=not RESULTS_PATH.exists(), index=False)


def save_jsonl(path, record):

    with path.open("a", encoding="utf-8") as file:
        file.write(json.dumps(record, ensure_ascii=False, default=str)+ "\n")

In [39]:
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
requests_attempted = 0
results_saved = 0

for start in range(0, len(run_queue), BUSINESSES_PER_REQUEST):

    batch = (run_queue
             .iloc[start:start + BUSINESSES_PER_REQUEST]
             .copy()
             .reset_index(drop=True))

    requests_attempted += 1
    batch_id = (f"{RUN_ID}_{requests_attempted}")

    try:
        prompt, response = verify_current_batch(batch)

        finish_reason = None
        finish_message = None

        if response.candidates:
            finish_reason = str(response.candidates[0].finish_reason)
            finish_message = response.candidates[0].finish_message

        search_queries, source_urls = (get_grounding_info(response))
        results, missing_numbers = (parse_response(response.text, batch, batch_id))

        # Save successful individual businesses
        save_results(results)
        results_saved += len(results)

        # Save the complete API request evidence
        save_jsonl(BATCH_LOG_PATH,
                   {
                       "BatchID": batch_id,
                       "Time": datetime.now().isoformat(timespec="seconds"),
                       "BusinessCount": len(batch),
                       "ParsedCount": len(results),
                       "BusinessNames": batch["BusinessName"].tolist(),
                       "RawResponse": response.text,
                       "SearchQueries": search_queries,
                       "SourceURLs": source_urls,
                       "Model": MODEL, 
                       "FinishReason": finish_reason,
                       "FinishMessage": finish_message
            })

        # Record anything Gemini failed to return properly
        if missing_numbers:
            missing_businesses = [batch.iloc[number - 1]["BusinessName"]
                                  for number in missing_numbers]

            save_jsonl(ERROR_LOG_PATH,
                       {
                           "BatchID": batch_id,
                           "ErrorType": "INCOMPLETE_RESPONSE",
                           "Businesses": missing_businesses
                           })

        print(f"Batch {requests_attempted}: saved {len(results)}/{len(batch)}")

    except errors.APIError as error:
        save_jsonl(ERROR_LOG_PATH,
                   {
                       "BatchID": batch_id,
                       "ErrorType": "API_ERROR",
                       "ErrorCode": error.code,
                       "ErrorMessage": error.message,
                       "Businesses": batch["BusinessName"].tolist()
                       })

        print(f"Batch {requests_attempted} failed: {error.code}")

        if error.code == 429:
            print("Rate limit reached. Stopping safely.")
            break

        if 400 <= error.code < 500:
            print("Non-retryable API error. Stopping safely.")
            break

    except Exception as error:
        save_jsonl(ERROR_LOG_PATH,
                   {
                       "BatchID": batch_id,
                       "ErrorType": type(error).__name__,
                       "ErrorMessage": str(error),
                       "Businesses": batch["BusinessName"].tolist()
                       })

        print(f"Batch {requests_attempted} failed: {error}")

    time.sleep(REQUEST_DELAY_SECONDS)

Batch 1: saved 10/10


In [40]:
if RESULTS_PATH.exists():

    all_results = pd.read_csv(RESULTS_PATH)

    print(
        f"""Requests attempted this run: {requests_attempted}
Results saved this run: {results_saved}
Total businesses completed: {len(all_results)}""")

    print(f"\nVerdicts: \n{all_results["AIVerdict"].value_counts()}")

Requests attempted this run: 1
Results saved this run: 10
Total businesses completed: 28847

Verdicts: 
AIVerdict
NOT_BAKERY    19875
UNCLEAR        6029
BAKERY         2943
Name: count, dtype: int64
